In [1]:
import ee
import geemap

PROJECT_ID = "project-gis-499303"

ee.Initialize(project=PROJECT_ID)

center_lon = 105.749433
center_lat = 21.039419

aoi = ee.Geometry.Polygon([
    [
        [105.7350, 21.0259],
        [105.7639, 21.0259],
        [105.7639, 21.0529],
        [105.7350, 21.0529],
        [105.7350, 21.0259]
    ]
])

Map = geemap.Map(center=[center_lat, center_lon], zoom=14)
Map.addLayer(aoi, {"color": "red"}, "AOI - Phenikaa Hospital Area")
Map

Matplotlib is building the font cache; this may take a moment.


Map(center=[21.039419, 105.749433], controls=(WidgetControl(options=['position', 'transparent_bg'], position='…

In [ ]:
def mask_s2_clouds(image):
    """
    Mask clouds and cloud shadows using Sentinel-2 SCL band.
    SCL = Scene Classification Layer.
    """
    scl = image.select("SCL")

    # Keep valid land/water pixels:
    # 4 = vegetation
    # 5 = bare soil / non-vegetated
    # 6 = water
    # 7 = unclassified
    # 11 = snow/ice, not relevant here but harmless
    valid_mask = (
        scl.eq(4)
        .Or(scl.eq(5))
        .Or(scl.eq(6))
        .Or(scl.eq(7))
        .Or(scl.eq(11))
    )

    return image.updateMask(valid_mask).divide(10000)


def get_s2_composite(start_date, end_date, aoi):
    collection = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(aoi)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 40))
        .map(mask_s2_clouds)
    )

    composite = collection.median().clip(aoi)
    return composite, collection.size()


before_start = "2018-10-01"
before_end = "2019-04-30"

after_start = "2024-10-01"
after_end = "2025-04-30"

before_img, before_count = get_s2_composite(before_start, before_end, aoi)
after_img, after_count = get_s2_composite(after_start, after_end, aoi)

print("Before image count:", before_count.getInfo())
print("After image count:", after_count.getInfo())

Before image count: 2
After image count: 6
